<a href="https://colab.research.google.com/github/specM7/DSGP_Group_33_Brain_Tumor_Predictor/blob/glioblastoma-Vidu-2425444/CNN_model_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import zipfile

zip_path = "/content/drive/MyDrive/archive.zip"
extract_path = "/content/extracted"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done extracting!")

Done extracting!


In [2]:
import shutil
import os

# Define the root path and the list of folders you want to remove
root_dir = "/content/extracted"
folders_to_remove = ["meningioma", "pituitary"] # Replace with your actual folder names

for folder in folders_to_remove:
    # Build the full path for Training and Testing
    train_path = os.path.join(root_dir, "Training", folder)
    test_path = os.path.join(root_dir, "Testing", folder)

    # Remove Training folder if it exists
    if os.path.exists(train_path):
        shutil.rmtree(train_path)
        print(f"Deleted: {train_path}")

    # Remove Testing folder if it exists
    if os.path.exists(test_path):
        shutil.rmtree(test_path)
        print(f"Deleted: {test_path}")

# Verify what remains
print("\nRemaining folders in Training:")
print(os.listdir(os.path.join(root_dir, "Training")))

Deleted: /content/extracted/Training/meningioma
Deleted: /content/extracted/Testing/meningioma
Deleted: /content/extracted/Training/pituitary
Deleted: /content/extracted/Testing/pituitary

Remaining folders in Training:
['glioma', 'notumor']


In [19]:
import tensorflow as tf
from tensorflow.keras import layers, models, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
import matplotlib.pyplot as plt

# 1. Configuration
IMG_SIZE = 128
BATCH_SIZE = 32
DATASET_PATH = "/content/processed_dataset" # Use your new padded dataset folder

# 2. Define Data Augmentation (Equivalent to your friend's logic)
data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2), # 20% rotation
    layers.RandomContrast(0.2), # ColorJitter equivalent
    layers.RandomZoom(0.1),
])

# 3. Load Data
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH + "/Training",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH + "/Testing",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

# Apply Augmentation only to Training set
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

# 4. Model Architecture
model = Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),

    Conv2D(32, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary output
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 5. Training with Early Stopping to prevent overfitting
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[early_stop])

# 6. Evaluation
model.save("final_model.h5")
print("Model saved as final_model.h5")

Found 2916 files belonging to 2 classes.
Found 705 files belonging to 2 classes.
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


92/92 ━━━━━━━━━━━━━━━━━━━━ 25s 205ms/step - accuracy: 0.9012 - loss: 0.9056 - val_accuracy: 0.4270 - val_loss: 5.4038
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9554 - loss: 0.1810 - val_accuracy: 0.4766 - val_loss: 4.5278
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9681 - loss: 0.1142 - val_accuracy: 0.5447 - val_loss: 4.6960
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9715 - loss: 0.1037 - val_accuracy: 0.7333 - val_loss: 2.0355
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9650 - loss: 0.1133 - val_accuracy: 0.9291 - val_loss: 0.3142
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9719 - loss: 0.1051 - val_accuracy: 0.9504 - val_loss: 0.1151
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9743 - loss: 0.1102 - val_accuracy: 0.8113 - val_loss: 1.5758
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9757 - loss: 0.0776 - val_accuracy: 0.9844 - val_loss: 

Model saved as final_model.h5


In [22]:
import tensorflow as tf
import numpy as np
from PIL import Image

print("Model expects:", model.input_shape)

def prepare_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((128, 128))  # IMPORTANT: 128
    img_array = np.array(img)     # DO NOT divide by 255
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

image_path = '/content/extracted/Testing/notumor/Te-noTr_0000.jpg'  # replace with your file
img_array = prepare_image(image_path)

print("Input shape:", img_array.shape)

predictions = model.predict(img_array)

# Instead of np.argmax, use a 0.5 threshold for your sigmoid output
predicted_class = 1 if predictions[0][0] > 0.5 else 0

print("Raw predictions:", predictions)
print("Predicted class:", predicted_class)


Model expects: (None, 128, 128, 3)
Input shape: (1, 128, 128, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Raw predictions: [[1.]]
Predicted class: 1


In [25]:
from sklearn.metrics import classification_report
import numpy as np

# 1. Collect all true labels and predicted labels
y_true = []
y_pred = []

# Iterate through the validation dataset
for images, labels in val_ds:
    # Get predictions
    preds = model.predict(images, verbose=0)
    # Threshold at 0.5 for binary classification
    binary_preds = (preds > 0.5).astype("int32")

    y_true.extend(labels.numpy())
    y_pred.extend(binary_preds.flatten())

# 2. Print the comprehensive report
print(classification_report(y_true, y_pred, target_names=['Glioma', 'No-Tumor']))

              precision    recall  f1-score   support

      Glioma       1.00      0.96      0.98       300
    No-Tumor       0.97      1.00      0.99       405

    accuracy                           0.98       705
   macro avg       0.99      0.98      0.98       705
weighted avg       0.98      0.98      0.98       705



In [11]:
import os
from PIL import Image

def check_image_dimensions(directory, expected_size=(128, 128)):
    inconsistent_files = []

    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, file)
                try:
                    with Image.open(img_path) as img:
                        if img.size != expected_size:
                            inconsistent_files.append((img_path, img.size))
                except Exception as e:
                    print(f"Error reading {file}: {e}")

    return inconsistent_files

# Run the check
path_to_check = "/content/processed_dataset/Testing"
mismatched = check_image_dimensions(path_to_check)

if mismatched:
    print(f"Found {len(mismatched)} images with incorrect dimensions:")
    for path, size in mismatched[:10]: # Print first 10
        print(f"{path}: {size}")
else:
    print("All images match the expected dimensions!")

All images match the expected dimensions!
